# 03 - Cleaning & Feature Engineering

## Objective
Turn the raw application table into a modelling frame: drop duplicates, null the
impossible values, and add ratio features that express credit risk in the way a
credit analyst would.

**Leakage rule applied here:** every engineered column is a function of a single
customer's own attributes. No column statistic (mean, quantile, target encoding)
is used, so these transformations are safe to run before the split. Everything
that *does* learn from the data -- imputation, scaling, encoding -- lives inside
the scikit-learn pipeline in notebook 05.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd

from src import config
from src.data import preprocessing
from src.features.engineering import FEATURE_DOCS, add_credit_risk_features, get_feature_lists

## Data loading and cleaning

In [ ]:
raw = preprocessing.load_raw_data()
TARGET = preprocessing.detect_target(raw)
clean, report = preprocessing.clean_data(raw, TARGET)
report

### Cleaning decisions

1. **Exact duplicates removed.** Identical rows are almost certainly repeated
   records; keeping them would over-weight those borrowers.
2. **Impossible values nulled, not clipped.** An age of 144 is not a real
   extreme value, it is an error; turning it into NaN lets the imputer handle it
   honestly instead of inventing a boundary value.
3. **Missing values left as NaN here.** They are imputed inside the pipeline
   (median for numeric, most frequent for categorical) so the imputation values
   are learned from the training fold only.
4. **Rows with an unusable target dropped.** A label cannot be imputed.
5. **Genuine outliers kept.** High incomes and large loans are real and informative.

In [ ]:
clean.isna().sum().to_frame("missing_after_clean")

## Feature engineering

In [ ]:
featured = add_credit_risk_features(clean)
new_columns = [c for c in featured.columns if c not in clean.columns]
pd.DataFrame({"feature": new_columns,
              "rationale": [FEATURE_DOCS.get(c, "") for c in new_columns]})

In [ ]:
featured[new_columns].describe().T.round(3)

### Do the new features separate the classes?

In [ ]:
engineered_numeric = [c for c in new_columns if pd.api.types.is_numeric_dtype(featured[c])]
featured.groupby(TARGET)[engineered_numeric].mean().T.round(3)

### Leakage self-check

* No engineered feature reads the target.
* No engineered feature uses another row's values.
* `loan_grade` and `loan_int_rate` are the lender's own risk assessment. They are
  legitimate inputs (known at decision time) but they make the model partly a
  *re-learner of the lender's policy* -- flagged in the README limitations.

In [ ]:
assert TARGET not in [c for c in new_columns]
numeric, categorical = get_feature_lists(featured, TARGET,
                                         drop=["cb_person_default_on_file"])
print(f"{len(numeric)} numeric features:\n{numeric}\n")
print(f"{len(categorical)} categorical features:\n{categorical}")

## Results

In [ ]:
preprocessing.save_dataframe(featured, config.FEATURED_DATA_FILE)
featured.head()

## Interpretation
The engineered set is deliberately small and interpretable: affordability ratios,
stability ratios and business-defined bands. Each one can be explained to a credit
officer in a single sentence.

## Conclusion
`data/processed/credit_risk_featured.csv` is the input for the SQL analytics
(notebook 04) and the modelling (notebook 05).